## Example 1: Creating and writing shapefiles

In [ ]:
import shapefile
import tempfile
import os

# Example 1: Creating and writing shapefiles
print("Example 1: Creating and writing shapefiles")

# Create a new shapefile writer
temp_dir = tempfile.mkdtemp()
shapefile_path = os.path.join(temp_dir, "test_points")

# Create writer for point geometry
w = shapefile.Writer(shapefile_path)

# Add fields to the shapefile
w.field('NAME', 'C', 40)  # Character field, max 40 chars
w.field('POPULATION', 'N', 10, 0)  # Numeric field, 10 digits, 0 decimal places
w.field('AREA', 'F', 13, 5)  # Float field, 13 digits, 5 decimal places

print(f"Shapefile writer created: {shapefile_path}")
print(f"Fields added: NAME, POPULATION, AREA")

# Add some sample points
points_data = [
    ("New York", [-74.0, 40.7], 8500000, 783.8),
    ("Los Angeles", [-118.2, 34.1], 4000000, 1302.9),
    ("Chicago", [-87.6, 41.9], 2700000, 606.3)
]

for name, coords, pop, area in points_data:
    w.point(coords[0], coords[1])  # Add point geometry
    w.record(name, pop, area)      # Add attribute record

print(f"Added {len(points_data)} points to shapefile")

# Close the writer to save files
w.close()

# Verify files were created
files_created = []
for ext in ['.shp', '.shx', '.dbf']:
    filepath = shapefile_path + ext
    if os.path.exists(filepath):
        size = os.path.getsize(filepath)
        files_created.append(f"{ext}: {size} bytes")

print(f"Files created: {files_created}")

# Cleanup
import shutil
shutil.rmtree(temp_dir)
print("Creating and writing shapefiles completed")

## Example 2: Reading and analyzing shapefiles

In [ ]:
import shapefile
import tempfile
import os

# Example 2: Reading and analyzing shapefiles
print("Example 2: Reading and analyzing shapefiles")

# Create a temporary shapefile first
temp_dir = tempfile.mkdtemp()
shapefile_path = os.path.join(temp_dir, "cities")

# Create sample data
w = shapefile.Writer(shapefile_path)
w.field('CITY', 'C', 30)
w.field('COUNTRY', 'C', 30) 
w.field('ELEVATION', 'N', 6, 0)

cities_data = [
    ("Paris", "France", [-2.3, 48.9], 35),
    ("Tokyo", "Japan", [139.7, 35.7], 40),
    ("Sydney", "Australia", [151.2, -33.9], 58),
    ("Cairo", "Egypt", [31.2, 30.0], 74),
    ("Denver", "USA", [-105.0, 39.7], 1655)
]

for city, country, coords, elevation in cities_data:
    w.point(coords[0], coords[1])
    w.record(city, country, elevation)

w.close()

# Now read the shapefile
sf = shapefile.Reader(shapefile_path)

print(f"Shapefile analysis:")
print(f"  File: {shapefile_path}")
print(f"  Shape type: {sf.shapeTypeName}")
print(f"  Number of records: {len(sf.records())}")

# Analyze fields
fields = sf.fields[1:]  # Skip deletion field
print(f"  Fields ({len(fields)} total):")
for field in fields:
    print(f"    {field[0]}: {field[1]} ({field[2]},{field[3]})")

# Read shapes and records
shapes = sf.shapes()
records = sf.records()

print(f"\nData analysis:")
for i, (shape, record) in enumerate(zip(shapes, records)):
    city, country, elevation = record
    x, y = shape.points[0]
    print(f"  {city}, {country}: ({x:.1f}, {y:.1f}) - {elevation}m elevation")

# Calculate bounding box
bbox = sf.bbox
print(f"\nBounding box: [{bbox[0]:.1f}, {bbox[1]:.1f}, {bbox[2]:.1f}, {bbox[3]:.1f}]")

# Cleanup
import shutil
shutil.rmtree(temp_dir)
print("Reading and analyzing shapefiles completed")

## Example 3: Working with polygon geometries

In [ ]:
import shapefile
import tempfile
import os

# Example 3: Working with polygon geometries
print("Example 3: Working with polygon geometries")

# Create a shapefile with polygon features
temp_dir = tempfile.mkdtemp()
poly_shapefile = os.path.join(temp_dir, "polygons")

w = shapefile.Writer(poly_shapefile)
w.field('NAME', 'C', 20)
w.field('AREA_CALC', 'F', 12, 4)

# Define some polygon shapes (rectangles and triangles)
polygons = [
    {
        'name': 'Rectangle1',
        'points': [[0, 0], [0, 2], [3, 2], [3, 0], [0, 0]],  # Closed polygon
        'area': 6.0
    },
    {
        'name': 'Triangle1', 
        'points': [[1, 1], [4, 1], [2.5, 4], [1, 1]],  # Triangle
        'area': 4.5
    },
    {
        'name': 'Rectangle2',
        'points': [[5, 0], [5, 1], [7, 1], [7, 0], [5, 0]],  # Small rectangle
        'area': 2.0
    }
]

print(f"Creating {len(polygons)} polygon features...")

for poly in polygons:
    w.poly([poly['points']])  # Create polygon geometry
    w.record(poly['name'], poly['area'])

w.close()

# Read back and analyze the polygons
sf = shapefile.Reader(poly_shapefile)
shapes = sf.shapes()
records = sf.records()

print(f"\nPolygon analysis:")
print(f"  Shape type: {sf.shapeTypeName}")
print(f"  Number of polygons: {len(shapes)}")

for i, (shape, record) in enumerate(zip(shapes, records)):
    name, area = record
    num_points = len(shape.points)
    bbox = shape.bbox
    
    print(f"\n  Polygon {i+1}: {name}")
    print(f"    Points: {num_points}")
    print(f"    Area (calculated): {area}")
    print(f"    Bounding box: [{bbox[0]:.1f}, {bbox[1]:.1f}, {bbox[2]:.1f}, {bbox[3]:.1f}]")
    print(f"    Parts: {len(shape.parts)}")

# Overall statistics
total_area = sum(record[1] for record in records)
print(f"\nSummary:")
print(f"  Total polygons: {len(records)}")
print(f"  Total area: {total_area}")

# Cleanup
import shutil
shutil.rmtree(temp_dir)
print("Working with polygon geometries completed")

## Example 4: Working with different geometry types and attributes

In [ ]:
import shapefile
import tempfile
import os
import sys
from io import StringIO

# Example 4: Working with different geometry types and attributes
print("Example 4: Working with different geometry types and attributes")

# Create shapefile with mixed geometry types (lines)
temp_dir = tempfile.mkdtemp()
line_shapefile = os.path.join(temp_dir, "roads")

w = shapefile.Writer(line_shapefile)
w.field('ROAD_NAME', 'C', 50)
w.field('SPEED_LIMIT', 'N', 3, 0)
w.field('ROAD_TYPE', 'C', 20)

# Define some line geometries (roads)
roads = [
    {
        'name': 'Main Street',
        'points': [[0, 0], [2, 1], [4, 0], [6, 2]],
        'speed': 35,
        'type': 'residential'
    },
    {
        'name': 'Highway 101', 
        'points': [[10, 5], [15, 8], [20, 10]],
        'speed': 65,
        'type': 'highway'
    },
    {
        'name': 'Oak Avenue',
        'points': [[1, 3], [3, 5], [5, 4]],
        'speed': 25,
        'type': 'local'
    }
]

print(f"Creating {len(roads)} road features...")

for road in roads:
    w.line([road['points']])  # Create line geometry
    w.record(road['name'], road['speed'], road['type'])

w.close()

# Read and analyze the line shapefile
sf = shapefile.Reader(line_shapefile)

print(f"\nRoad network analysis:")
print(f"  Shape type: {sf.shapeTypeName}")
print(f"  Number of roads: {len(sf.records())}")

shapes = sf.shapes()
records = sf.records()

for i, (shape, record) in enumerate(zip(shapes, records)):
    name, speed, road_type = record
    num_points = len(shape.points)
    
    # Calculate approximate road length
    length = 0
    points = shape.points
    for j in range(1, len(points)):
        dx = points[j][0] - points[j-1][0]
        dy = points[j][1] - points[j-1][1]
        length += (dx**2 + dy**2)**0.5
    
    print(f"  {name}: {road_type} road, {speed} mph, {num_points} points, ~{length:.1f} units long")

# Cleanup
import shutil
shutil.rmtree(temp_dir)
print("Working with different geometry types completed")

## Example 5: Shapefile operations and error handling

In [ ]:
import shapefile
import tempfile
import os

# Example 5: Shapefile operations and error handling
print("Example 5: Shapefile operations and error handling")

# Create a more complex shapefile with validation
temp_dir = tempfile.mkdtemp()
complex_shapefile = os.path.join(temp_dir, "land_parcels")

try:
    w = shapefile.Writer(complex_shapefile)
    
    # Add multiple field types to test shapefile capabilities
    w.field('PARCEL_ID', 'N', 10, 0)
    w.field('OWNER_NAME', 'C', 100)
    w.field('AREA_SQFT', 'F', 15, 2)
    w.field('ZONING', 'C', 10)
    w.field('BUILDABLE', 'L')  # Logical/Boolean field
    
    # Create complex polygon parcels
    parcels = [
        {
            'id': 1001,
            'owner': 'Smith, John',
            'area': 8520.50,
            'zoning': 'R1',
            'buildable': True,
            'boundary': [[0, 0], [0, 100], [85, 100], [85, 0], [0, 0]]
        },
        {
            'id': 1002, 
            'owner': 'Johnson Real Estate LLC',
            'area': 12450.75,
            'zoning': 'C2',
            'buildable': True,
            'boundary': [[100, 0], [100, 150], [183, 150], [183, 0], [100, 0]]
        },
        {
            'id': 1003,
            'owner': 'City Parks Department',
            'area': 25000.00,
            'zoning': 'OS',
            'buildable': False,
            'boundary': [[200, 0], [200, 200], [325, 200], [325, 0], [200, 0]]
        }
    ]
    
    print(f"Writing {len(parcels)} land parcels...")
    
    for parcel in parcels:
        # Validate polygon closure
        boundary = parcel['boundary']
        if boundary[0] != boundary[-1]:
            print(f"Warning: Parcel {parcel['id']} boundary not closed, auto-closing")
            boundary.append(boundary[0])
        
        w.poly([boundary])
        w.record(
            parcel['id'],
            parcel['owner'], 
            parcel['area'],
            parcel['zoning'],
            parcel['buildable']
        )
    
    w.close()
    
    # Validate the created shapefile
    sf = shapefile.Reader(complex_shapefile)
    
    print(f"\nShapefile validation:")
    print(f"  File: {complex_shapefile}")
    print(f"  Shape type: {sf.shapeTypeName}")
    print(f"  Record count: {len(sf.records())}")
    print(f"  Field count: {len(sf.fields) - 1}")  # -1 for deletion field
    
    # Detailed field analysis
    print(f"\nField definitions:")
    for field in sf.fields[1:]:  # Skip deletion field
        print(f"  {field[0]}: {field[1]} ({field[2]},{field[3]})")
    
    # Calculate summary statistics
    records = sf.records()
    total_area = sum(float(record[2]) for record in records)
    buildable_count = sum(1 for record in records if record[4])
    
    print(f"\nSummary statistics:")
    print(f"  Total area: {total_area:,.2f} sq ft")
    print(f"  Buildable parcels: {buildable_count}/{len(records)}")
    
    # Group by zoning
    zoning_stats = {}
    for record in records:
        zoning = record[3]
        area = float(record[2])
        if zoning not in zoning_stats:
            zoning_stats[zoning] = {'count': 0, 'area': 0}
        zoning_stats[zoning]['count'] += 1
        zoning_stats[zoning]['area'] += area
    
    print(f"\nZoning breakdown:")
    for zone, stats in zoning_stats.items():
        print(f"  {zone}: {stats['count']} parcels, {stats['area']:,.2f} sq ft")
    
except Exception as e:
    print(f"Error processing shapefile: {e}")
    raise AssertionError(
        f"pyshp example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e
    
finally:
    # Cleanup
    import shutil
    if os.path.exists(temp_dir):
        shutil.rmtree(temp_dir)

print("Shapefile operations completed")